# 03 - Results Analysis

Comprehensive analysis of trained models:
- Accuracy comparison across encodings
- Confusion matrices
- Per-class accuracy analysis
- Energy comparison (SNN vs ANN)
- Statistical significance

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.config import RESULTS_DIR, NUM_STEPS, get_device
from src.dataset import download_esc50, get_fold_dataloaders, get_class_labels
from src.encoding import get_encoder
from src.evaluate import (
    evaluate_experiment, generate_comparison_table,
    load_predictions, compute_fold_metrics,
)
from src.energy import compare_energy, save_energy_report
from src.models.snn_model import SpikingCNN
from src.models.ann_model import ConvANN

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Run Evaluation for All Experiments

In [ ]:
# Evaluate all trained experiments
experiments = [
    ('ann', 'none'),
    ('snn', 'rate'),
    ('snn', 'delta'),
    ('snn', 'latency'),
    ('snn', 'direct'),
]

for model, enc in experiments:
    try:
        evaluate_experiment(model, enc)
    except FileNotFoundError:
        print(f"  Skipping {model}/{enc} (not trained yet)")

## 2. Comparison Table

In [ ]:
rows = generate_comparison_table(experiments)

In [ ]:
# Bar chart comparison
if rows:
    df = pd.DataFrame(rows)
    df['label'] = df['model'].str.upper() + ' / ' + df['encoding']
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(df['label'], df['mean_accuracy'], yerr=df['std_accuracy'],
                  capsize=5, color=['#e74c3c' if m == 'ann' else '#3498db' for m in df['model']])
    ax.set_ylabel('Mean Accuracy (5-fold CV)')
    ax.set_title('ESC-50 Classification Accuracy: SNN vs ANN')
    ax.set_ylim(0, 1)
    
    for bar, acc in zip(bars, df['mean_accuracy']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{acc:.1%}', ha='center', fontsize=10)
    
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'accuracy_comparison.png'), dpi=150)
    plt.show()

## 3. Energy Comparison

In [ ]:
device = get_device()
download_esc50()

# Load a test batch for energy measurement
_, test_loader = get_fold_dataloaders(1, batch_size=16)
sample_data, _ = next(iter(test_loader))
sample_data = sample_data.to(device)

# Load best SNN model (try rate encoding first)
best_snn_path = RESULTS_DIR / 'snn' / 'rate' / 'best_fold1.pt'
best_ann_path = RESULTS_DIR / 'ann' / 'none' / 'best_fold1.pt'

snn_model = SpikingCNN().to(device)
ann_model = ConvANN().to(device)

if best_snn_path.exists():
    snn_model.load_state_dict(torch.load(best_snn_path, map_location=device, weights_only=True))
if best_ann_path.exists():
    ann_model.load_state_dict(torch.load(best_ann_path, map_location=device, weights_only=True))

# Encode for SNN
encoder = get_encoder('rate')
snn_input = encoder(sample_data).to(device)

# Run energy comparison
energy_result = compare_energy(snn_model, ann_model, snn_input, sample_data)
save_energy_report(energy_result)

In [ ]:
# Energy comparison bar chart
fig, ax = plt.subplots(figsize=(8, 5))
models = ['ANN', 'SNN']
energies = [energy_result['ann']['energy_pJ'], energy_result['snn']['energy_pJ']]
colors = ['#e74c3c', '#3498db']

bars = ax.bar(models, energies, color=colors)
ax.set_ylabel('Energy per sample (pJ)')
ax.set_title('Energy Comparison: SNN vs ANN\n(SynOps on Loihi vs MACs on 45nm CMOS)')

for bar, e in zip(bars, energies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(energies)*0.02,
            f'{e:,.0f} pJ', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'energy' / 'energy_comparison.png'), dpi=150)
plt.show()

print(f"\nSNN uses {energy_result['energy_ratio']:.2%} of ANN energy")

## 4. Statistical Significance

In [ ]:
# Paired t-test: best SNN encoding vs ANN across 5 folds
ann_path = RESULTS_DIR / 'ann' / 'none' / 'summary.json'
snn_rate_path = RESULTS_DIR / 'snn' / 'rate' / 'summary.json'

if ann_path.exists() and snn_rate_path.exists():
    with open(ann_path) as f:
        ann_data = json.load(f)
    with open(snn_rate_path) as f:
        snn_data = json.load(f)
    
    ann_accs = ann_data['fold_accuracies']
    snn_accs = snn_data['fold_accuracies']
    
    t_stat, p_value = stats.ttest_rel(ann_accs, snn_accs)
    
    print(f"ANN fold accuracies: {[f'{a:.4f}' for a in ann_accs]}")
    print(f"SNN fold accuracies: {[f'{a:.4f}' for a in snn_accs]}")
    print(f"\nPaired t-test: t={t_stat:.4f}, p={p_value:.4f}")
    print(f"{'Significant' if p_value < 0.05 else 'Not significant'} at p<0.05")
else:
    print("Results not available yet. Train models first.")

## 5. Per-Class Deep Dive

In [ ]:
# Which classes are hardest/easiest for SNN vs ANN?
class_labels = get_class_labels()

ann_eval_path = RESULTS_DIR / 'ann' / 'none' / 'evaluation.json'
snn_eval_path = RESULTS_DIR / 'snn' / 'rate' / 'evaluation.json'

if ann_eval_path.exists() and snn_eval_path.exists():
    with open(ann_eval_path) as f:
        ann_eval = json.load(f)
    with open(snn_eval_path) as f:
        snn_eval = json.load(f)
    
    # Compare per-class accuracy
    df_compare = pd.DataFrame({
        'class': list(ann_eval['per_class_accuracy'].keys()),
        'ann_acc': list(ann_eval['per_class_accuracy'].values()),
        'snn_acc': list(snn_eval['per_class_accuracy'].values()),
    })
    df_compare['diff'] = df_compare['snn_acc'] - df_compare['ann_acc']
    df_compare = df_compare.sort_values('diff')
    
    # Plot: SNN advantage vs disadvantage per class
    fig, ax = plt.subplots(figsize=(10, 14))
    colors = ['#d32f2f' if d < 0 else '#388e3c' for d in df_compare['diff']]
    ax.barh(df_compare['class'], df_compare['diff'], color=colors)
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set_xlabel('SNN accuracy - ANN accuracy')
    ax.set_title('Per-Class: Where SNN Beats/Loses to ANN')
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'snn_vs_ann_per_class.png'), dpi=150)
    plt.show()
    
    print("\nTop 5 classes where SNN outperforms ANN:")
    print(df_compare.tail(5)[['class', 'ann_acc', 'snn_acc', 'diff']].to_string(index=False))
    print("\nTop 5 classes where ANN outperforms SNN:")
    print(df_compare.head(5)[['class', 'ann_acc', 'snn_acc', 'diff']].to_string(index=False))
else:
    print("Results not available yet. Train and evaluate models first.")